# RAG Embedding 생성 (Google Colab GPU)

이 노트북은 `chunks.jsonl`의 텍스트를 **BAAI/bge-m3** 모델로 임베딩합니다.

**모델:** BAAI/bge-m3 (1024차원, 다국어/한국어 우수)

**사전 준비:** Google Drive에 `chunks.jsonl`을 업로드해두세요.

**워크플로우:**
1. Google Drive 마운트 → chunks.jsonl 읽기
2. GPU로 임베딩 생성 (enriched_chunk_text 사용)
3. `embeddings.npz`를 Drive에 저장
4. 로컬에서 Drive의 `embeddings.npz` 다운로드 후 `upload_to_pgvector.py` 실행

**런타임 설정:** `런타임 > 런타임 유형 변경 > GPU (T4)` 선택

In [1]:
# 1. 의존성 설치
!pip install -q langchain-huggingface torch tqdm numpy

In [2]:
# 2. GPU 확인
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

CUDA available: True
GPU: Tesla T4
Memory: 14.6 GB


In [3]:
# 3. Google Drive 마운트 및 경로 설정
from google.colab import drive
drive.mount('/content/drive')

# Google Drive 내 파일 경로 설정 (필요시 수정)
DRIVE_DIR = "/content/drive/MyDrive/rag-practice"
CHUNKS_PATH = f"{DRIVE_DIR}/chunks.jsonl"
OUTPUT_PATH = f"{DRIVE_DIR}/embeddings.npz"

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
assert os.path.exists(CHUNKS_PATH), f"파일을 찾을 수 없습니다: {CHUNKS_PATH}\nGoogle Drive의 '{DRIVE_DIR}' 폴더에 chunks.jsonl을 업로드해주세요."
print(f"Input: {CHUNKS_PATH}")
print(f"Output: {OUTPUT_PATH}")

Mounted at /content/drive
Input: /content/drive/MyDrive/rag-practice/chunks.jsonl
Output: /content/drive/MyDrive/rag-practice/embeddings.npz


In [4]:
# 4. 청크 로드 및 임베딩 대상 준비
import json

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

MIN_CHUNK_LENGTH = 20
texts, metadata_list = [], []

for i, chunk in enumerate(chunks):
    # enriched_chunk_text로 임베딩 (컨텍스트 포함), 없으면 chunk_text 사용
    page_content = chunk.get("enriched_chunk_text", chunk.get("chunk_text", "")).strip()
    original = chunk.get("chunk_text", "").strip()

    if len(original) > MIN_CHUNK_LENGTH:
        texts.append(page_content)
        metadata_list.append({
            "id": chunk.get("id", str(i)),
            "chunk_index": chunk.get("chunk_index", 0),
            "title": chunk.get("title", ""),
            "url": chunk.get("url", ""),
            "source_type": chunk.get("source_type", "Unknown"),
            "original_chunk_text": original,
        })

print(f"Total texts to embed: {len(texts)}")
print(f"Sample enriched text:\n{texts[0][:200]}...")

Total texts to embed: 4433
Sample enriched text:
[문서: 세종]
세종(한국 한자: 世宗, 중세 한국어: ·솅조ᇰ, 1397년 5월 15일 (음력 4월 10일)~1450년 3월 30일 (음력 2월 17일))은 조선의 제4대 국왕(재위 : 1418년 9월 9일~1450년 3월 30일)으로, 태종과 원경왕후의 아들이다. 형인 양녕대군이 폐세자가 되자 세자에 책봉되었으며 태종의 양위를 받아 즉위하였다....


In [5]:
# 5. 임베딩 모델 로드 (GPU)
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "BAAI/bge-m3"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cuda",
        "model_kwargs": {"torch_dtype": torch.float16}
    }
)
print(f"Embedding model loaded: {EMBEDDING_MODEL} on GPU")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model loaded: BAAI/bge-m3 on GPU


In [6]:
# 6. 배치 임베딩 생성
import numpy as np
from tqdm import tqdm

BATCH_SIZE = 32  # bge-m3는 Qwen보다 크므로 배치 크기를 줄임

all_embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding batches"):
    batch = texts[i:i + BATCH_SIZE]
    embs = embedding_model.embed_documents(batch)
    all_embeddings.extend(embs)
    torch.cuda.empty_cache()

embeddings_array = np.array(all_embeddings, dtype=np.float32)
print(f"Embeddings shape: {embeddings_array.shape}")  # (N, 1024)

Embedding batches: 100%|██████████| 139/139 [01:41<00:00,  1.37it/s]

Embeddings shape: (4433, 1024)


In [7]:
# 7. embeddings.npz를 Google Drive에 저장
np.savez_compressed(
    OUTPUT_PATH,
    embeddings=embeddings_array,
    texts=np.array(texts, dtype=object),
    metadata=np.array(
        [json.dumps(m, ensure_ascii=False) for m in metadata_list],
        dtype=object
    )
)

file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"embeddings.npz 저장 완료 ({file_size:.1f} MB)")
print(f"  - {len(texts)} vectors x {embeddings_array.shape[1]} dimensions")
print(f"  - 저장 위치: {OUTPUT_PATH}")
print(f"\nGoogle Drive에서 embeddings.npz를 다운로드한 후:")
print(f"  python upload_to_pgvector.py --embeddings data/embeddings.npz")

embeddings.npz 저장 완료 (12.6 MB)
  - 4433 vectors x 1024 dimensions
  - 저장 위치: /content/drive/MyDrive/rag-practice/embeddings.npz

Google Drive에서 embeddings.npz를 다운로드한 후:
  python upload_to_pgvector.py --embeddings data/embeddings.npz


## 다음 단계

1. Google Drive에서 `embeddings.npz`를 로컬 `packages/rag-data/data/` 폴더로 다운로드
2. 로컬 터미널에서 실행:

```bash
cd packages/rag-data
python upload_to_pgvector.py --embeddings data/embeddings.npz
```

이 스크립트가 PGVector에 벡터를 업로드합니다. 업로드 후 백엔드 서버를 재시작하세요.